In [1]:
import re
import asyncio
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError

START_URL = "https://www.gaswork.com/search/Anesthesiologist/Job/All"
DESKTOP = Path.home() / "Desktop"

COLUMN_ORDER = [
    "Compare",
    "Ref #",
    "City",
    "State",
    "Brief Description",
    "Duration",
    "Min $",
    "Max $",
    "W-2",
    "User Type",
    "Company",
    "Updated",
    "Date Posted",
    "Job Link",
]

def clean_text(value: str) -> str:
    return re.sub(r"\s+", " ", str(value or "")).strip()

def ensure_desktop():
    DESKTOP.mkdir(parents=True, exist_ok=True)

def parse_results_counter(text: str):
    text = clean_text(text)
    m = re.search(r"SEARCH RESULTS:\s*\((\d+)\s*-\s*(\d+)\s+of\s+(\d+)\)", text, re.I)
    if m:
        return int(m.group(1)), int(m.group(2)), int(m.group(3))
    m = re.search(r"SEARCH RESULTS:\s*(\d+)\s*-\s*(\d+)\s+of\s+(\d+)", text, re.I)
    if m:
        return int(m.group(1)), int(m.group(2)), int(m.group(3))
    return None, None, None

def is_rate_limited(text: str) -> bool:
    text = clean_text(text).lower()
    signals = [
        "maximum number of search queries allowed",
        "try again later",
        "exceeded the maximum number of search queries",
    ]
    return any(s in text for s in signals)

async def save_debug(page, prefix="gaswork_debug"):
    ensure_desktop()
    html_path = DESKTOP / f"{prefix}.html"
    png_path = DESKTOP / f"{prefix}.png"

    try:
        html_path.write_text(await page.content(), encoding="utf-8")
    except Exception:
        pass

    try:
        await page.screenshot(path=str(png_path), full_page=True)
    except Exception:
        pass

    print(f"Saved debug HTML: {html_path}")
    print(f"Saved debug PNG:  {png_path}")

async def launch_browser():
    p = await async_playwright().start()

    browser = await p.chromium.launch(
        headless=False,
        args=["--disable-blink-features=AutomationControlled"],
    )

    context = await browser.new_context(
        viewport={"width": 1440, "height": 1000},
        user_agent=(
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        ),
        locale="en-US",
        java_script_enabled=True,
    )

    page = await context.new_page()

    await page.add_init_script("""
        Object.defineProperty(navigator, 'webdriver', {
            get: () => undefined
        });
    """)

    return p, browser, context, page

async def open_page(page, url):
    response = await page.goto(url, wait_until="domcontentloaded", timeout=120000)
    await page.wait_for_timeout(4000)

    body = await page.locator("body").inner_text()
    title = await page.title()
    start_n, end_n, total_n = parse_results_counter(body)

    print("Status:", response.status if response else None)
    print("URL:", page.url)
    print("Title:", title)
    print("Counter:", (start_n, end_n, total_n))

    return {
        "status": response.status if response else None,
        "url": page.url,
        "title": title,
        "body": body,
        "start_n": start_n,
        "end_n": end_n,
        "total_n": total_n,
    }

async def try_set_100_results(page):
    print("Trying to increase results per page...")
    for text in [
        "Show 100 results per page",
        "100 results per page",
        "Show 75 results per page",
        "Show 50 results per page",
    ]:
        try:
            loc = page.get_by_text(text, exact=False).first
            if await loc.count() > 0:
                await loc.scroll_into_view_if_needed()
                await loc.click(timeout=5000)
                await page.wait_for_timeout(5000)
                body = await page.locator("body").inner_text()
                print("Counter after page-size click:", parse_results_counter(body))
                return
        except Exception as e:
            print(f"Could not click '{text}': {e}")

    print("Could not change results per page. Continuing anyway.")

async def extract_rows(page):
    raw_rows = await page.locator("tr").evaluate_all("""
        (trs) => trs.map(tr => {
            const cells = [...tr.querySelectorAll("td")].map(td =>
                (td.innerText || "").replace(/\\s+/g, " ").trim()
            );
            const links = [...tr.querySelectorAll("a")].map(a => ({
                text: (a.innerText || "").replace(/\\s+/g, " ").trim(),
                href: a.getAttribute("href") || ""
            }));
            return {
                row_text: (tr.innerText || "").replace(/\\s+/g, " ").trim(),
                cells,
                links
            };
        })
    """)

    parsed = []

    for r in raw_rows:
        row_text = clean_text(r.get("row_text", ""))
        cells = [clean_text(x) for x in r.get("cells", [])]
        links = r.get("links", [])

        if not row_text:
            continue

        ref_number = ""
        href = ""

        for link in links:
            link_text = clean_text(link.get("text", ""))
            if re.fullmatch(r"\d{5,7}", link_text):
                ref_number = link_text
                href = link.get("href", "")
                break

        if not ref_number:
            m = re.match(r"^(\d{5,7})\b", row_text)
            if m:
                ref_number = m.group(1)

        if not ref_number:
            continue

        row = {
            "Compare": "",
            "Ref #": ref_number,
            "City": "",
            "State": "",
            "Brief Description": "",
            "Duration": "",
            "Min $": "",
            "Max $": "",
            "W-2": "",
            "User Type": "",
            "Company": "",
            "Updated": "",
            "Date Posted": "",
            "Job Link": urljoin(page.url, href) if href else "",
        }

        if len(cells) >= 10:
            padded = cells[:]
            while len(padded) < 13:
                padded.append("")

            row["Compare"] = padded[0]
            row["City"] = padded[2] if len(padded) > 2 else ""
            row["State"] = padded[3] if len(padded) > 3 else ""
            row["Brief Description"] = padded[4] if len(padded) > 4 else ""
            row["Duration"] = padded[5] if len(padded) > 5 else ""
            row["Min $"] = padded[6] if len(padded) > 6 else ""
            row["Max $"] = padded[7] if len(padded) > 7 else ""
            row["W-2"] = padded[8] if len(padded) > 8 else ""
            row["User Type"] = padded[9] if len(padded) > 9 else ""
            row["Company"] = padded[10] if len(padded) > 10 else ""
            row["Updated"] = padded[11] if len(padded) > 11 else ""
            row["Date Posted"] = padded[12] if len(padded) > 12 else ""
        else:
            row["Brief Description"] = row_text

        parsed.append(row)

    deduped = {}
    for row in parsed:
        deduped[row["Ref #"]] = row

    return list(deduped.values())

async def click_next_safe(page, old_end_n):
    """
    Click next, then inspect page text manually.
    No wait_for_function, so no TargetClosedError chain.
    """
    try:
        old_body = await page.locator("body").inner_text()
    except Exception:
        return False, None, "page_unavailable"

    clicked = False

    for loc in [
        page.locator("a", has_text="next").last,
        page.get_by_text("next", exact=False).last,
        page.get_by_text("Next", exact=False).last,
    ]:
        try:
            if await loc.count() > 0:
                await loc.scroll_into_view_if_needed()
                await loc.click(timeout=5000)
                clicked = True
                break
        except Exception:
            pass

    if not clicked:
        return False, None, "no_next"

    # slow down to reduce rate limiting
    await page.wait_for_timeout(6000)

    try:
        new_body = await page.locator("body").inner_text()
    except Exception:
        return False, None, "page_closed"

    if is_rate_limited(new_body):
        return False, parse_results_counter(new_body), "rate_limited"

    new_counter = parse_results_counter(new_body)

    if new_body != old_body and new_counter[1] is not None and old_end_n is not None:
        if new_counter[1] > old_end_n:
            return True, new_counter, "advanced"

    if new_body != old_body and new_counter != (None, None, None):
        return True, new_counter, "changed"

    return False, new_counter, "stale"

async def scrape_gaswork(file_base="gaswork_jobs_all"):
    ensure_desktop()

    p, browser, context, page = await launch_browser()
    all_rows = []
    seen_refs = set()

    try:
        info = await open_page(page, START_URL)

        if is_rate_limited(info["body"]):
            print("Rate limited immediately.")
            await save_debug(page, "gaswork_rate_limited_start")
            return pd.DataFrame(columns=COLUMN_ORDER)

        if info["total_n"] is None:
            print("Could not detect the results counter.")
            await save_debug(page, "gaswork_counter_missing")
            return pd.DataFrame(columns=COLUMN_ORDER)

        await try_set_100_results(page)

        expected_total = None
        page_num = 1
        stale_count = 0

        while True:
            try:
                body = await page.locator("body").inner_text()
            except Exception:
                print("Page no longer available.")
                break

            if is_rate_limited(body):
                print("Rate limited by GasWork. Stopping and saving partial data.")
                await save_debug(page, f"gaswork_rate_limited_at_{len(seen_refs)}")
                break

            start_n, end_n, total_n = parse_results_counter(body)

            if expected_total is None and total_n is not None:
                expected_total = total_n

            print(f"\n--- Page {page_num} ---")
            print("Counter on page:", (start_n, end_n, total_n))
            print("Total unique rows so far:", len(seen_refs))

            rows = await extract_rows(page)
            print("Parsed rows on page:", len(rows))

            if not rows:
                print("No rows parsed.")
                await save_debug(page, f"gaswork_empty_page_{page_num}")
                break

            before = len(seen_refs)
            for row in rows:
                ref = row["Ref #"]
                if ref and ref not in seen_refs:
                    seen_refs.add(ref)
                    all_rows.append(row)

            added = len(seen_refs) - before
            print("New rows added:", added)
            print("Total unique rows:", len(seen_refs))

            if expected_total is not None and len(seen_refs) >= expected_total:
                print("Reached expected total.")
                break

            if end_n is not None and total_n is not None and end_n >= total_n:
                print("Counter indicates final page reached.")
                break

            moved, new_counter, status = await click_next_safe(page, end_n)
            print("Next status:", status)
            print("Counter after next:", new_counter)

            if status == "rate_limited":
                print("GasWork rate-limited the session. Saving partial data.")
                await save_debug(page, f"gaswork_rate_limited_at_{len(seen_refs)}")
                break

            if not moved:
                stale_count += 1
                if stale_count >= 2:
                    print("Stopping after repeated next failures.")
                    await save_debug(page, f"gaswork_stopped_at_{len(seen_refs)}")
                    break
            else:
                stale_count = 0
                page_num += 1

            # extra pause between pages to reduce throttling
            await asyncio.sleep(4)

        df = pd.DataFrame(all_rows)

        if df.empty:
            print("No rows collected.")
            return pd.DataFrame(columns=COLUMN_ORDER)

        for col in COLUMN_ORDER:
            if col not in df.columns:
                df[col] = ""

        df = df[COLUMN_ORDER].drop_duplicates(subset=["Ref #"]).reset_index(drop=True)

        csv_path = DESKTOP / f"{file_base}.csv"
        xlsx_path = DESKTOP / f"{file_base}.xlsx"

        df.to_csv(csv_path, index=False)
        df.to_excel(xlsx_path, index=False)

        print("\nDONE")
        print("Final total jobs collected:", len(df))
        print("Expected total from page:", expected_total)
        print("CSV saved to:", csv_path)
        print("Excel saved to:", xlsx_path)

        return df

    finally:
        try:
            await context.close()
        except Exception:
            pass
        try:
            await browser.close()
        except Exception:
            pass
        try:
            await p.stop()
        except Exception:
            pass

# RUN
df = await scrape_gaswork(file_base="gaswork_jobs_all")
display(df.head(20))
print(df.shape)

Status: 200
URL: https://www.gaswork.com/search/Anesthesiologist/Job/All
Title: GasWork.com - Search - Anesthesiologist Jobs
Counter: (1, 25, 4278)
Trying to increase results per page...
Could not click 'Show 100 results per page': Locator.scroll_into_view_if_needed: Timeout 29953ms exceeded.
Call log:
  - attempting scroll into view action
    2 × waiting for element to be stable
      - element is not visible
    - retrying scroll into view action
    - waiting 20ms
    2 × waiting for element to be stable
      - element is not visible
    - retrying scroll into view action
      - waiting 100ms
    56 × waiting for element to be stable
       - element is not visible
     - retrying scroll into view action
       - waiting 500ms

Could not click '100 results per page': Locator.scroll_into_view_if_needed: Timeout 29985ms exceeded.
Call log:
  - attempting scroll into view action
    2 × waiting for element to be stable
      - element is not visible
    - retrying scroll into view a

,Compare,Ref #,City,State,Brief Description,Duration,Min $,Max $,W-2,User Type,Company,Updated,Date Posted,Job Link
0,,317785,317785,Dallas,Texas,Pediatric Cardiac Anesthesiologist needed for ...,Full Time,"$700,000","$750,000",W-2,Group: National Private Practice,US Anesthesia Partners (USAP)- Medical City Da...,04/16/2026,https://www.gaswork.com/post/317785
1,,334312,334312,"Frisco, Plano, and Dallas!",Texas,Seeking Cardiac or General full-time positions...,Full Time,"$500,000","$600,000",W-2,Group: National Private Practice,U.S. Anesthesia Partners- Excel Division,04/16/2026,https://www.gaswork.com/post/334312
2,,553005,553005,Dallas,Texas,USAP is now looking for a Physician to serve a...,Full Time,,,W-2,Group: National Private Practice,U.S. Anesthesia Partners- Medical City Dallas-...,04/16/2026,https://www.gaswork.com/post/553005
3,,372324,372324,Dallas,Texas,Customizable practice opportunity in Dallas! H...,Full Time,"$525,000",,W-2,Group: National Private Practice,US Anesthesia Partners (USAP)- PAP Division,04/16/2026,https://www.gaswork.com/post/372324
4,,431247,431247,San Antonio,Texas,100% Specialty or choose your own hybrid Peds/...,Full Time,,,W-2,Group: National Private Practice,US Anesthesia Partners (USAP),04/16/2026,https://www.gaswork.com/post/431247
5,,572739,572739,Roslyn,New York,"General & Cardiac need at high-volume, multi-s...",Full Time,"$650,000",,W-2,Group: Private Practice,NYCA,04/16/2026,https://www.gaswork.com/post/572739
6,,496901,496901,Oshkosh,Wisconsin,"Aurora Medical Center in Oshkosh, WI seeking a...",Full Time,"$550,000",,W-2,Facility: Hospital,Advocate Aurora Health,04/16/2026,https://www.gaswork.com/post/496901
7,,559190,559190,Modesto,California,1099 Opportunities in Both Hospital and ASC Se...,Full Time,"$390,000","$775,000",1099,Group: National Private Practice,North American Partners in Anesthesia,04/16/2026,https://www.gaswork.com/post/559190
8,,406196,406196,Roseburg,Oregon,Substantial Sign On Bonus! $600K+/yr for Full ...,Full Time,"$600,000",,1099,Group: National Private Practice,Vituity,04/16/2026,https://www.gaswork.com/post/406196
9,,522113,522113,San Jose,California,Join a supportive team at a high acuity practi...,Full Time,,,,Group: National Private Practice,Vituity,04/16/2026,https://www.gaswork.com/post/522113


(3733, 14)


In [3]:
# =========================
# RESUME PATCH (BOTTOM CELL)
# =========================

from pathlib import Path
import pandas as pd
import asyncio

DESKTOP = Path.home() / "Desktop"
EXISTING_FILE = DESKTOP / "gaswork_jobs_all.csv"

# sanity check so the error is clear if the main cell was not run
required = [
    "ensure_desktop",
    "launch_browser",
    "open_page",
    "try_set_100_results",
    "extract_rows",
    "click_next_safe",
    "parse_results_counter",
    "is_rate_limited",
    "COLUMN_ORDER",
    "START_URL",
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Run your main scraper cell first. Missing: " + ", ".join(missing)
    )

async def scrape_gaswork_resume(file_base="gaswork_jobs_all"):
    ensure_desktop()

    # load what you already have
    existing_refs = set()
    if EXISTING_FILE.exists():
        print("Loading existing file...")
        existing_df = pd.read_csv(EXISTING_FILE)
        existing_df["Ref #"] = existing_df["Ref #"].astype(str)
        existing_refs = set(existing_df["Ref #"])
        print("Already have:", len(existing_refs))
    else:
        existing_df = pd.DataFrame(columns=COLUMN_ORDER)

    p, browser, context, page = await launch_browser()

    seen_refs = set(existing_refs)
    all_rows = existing_df.to_dict("records") if not existing_df.empty else []

    try:
        info = await open_page(page, START_URL)

        if is_rate_limited(info["body"]):
            print("Rate limited immediately.")
            return existing_df

        if info["total_n"] is None:
            print("Could not detect results counter.")
            return existing_df

        await try_set_100_results(page)

        page_num = 1
        stale_count = 0

        while True:
            try:
                body = await page.locator("body").inner_text()
            except Exception:
                print("Page no longer available.")
                break

            if is_rate_limited(body):
                print("Hit rate limit. Saving progress and stopping.")
                break

            start_n, end_n, total_n = parse_results_counter(body)

            print(f"\n--- Page {page_num} ---")
            print("Counter:", (start_n, end_n, total_n))
            print("Current total:", len(seen_refs))

            rows = await extract_rows(page)
            if not rows:
                print("No rows parsed.")
                break

            before = len(seen_refs)

            for row in rows:
                ref = str(row["Ref #"])
                if ref and ref not in seen_refs:
                    seen_refs.add(ref)
                    row["Ref #"] = ref
                    all_rows.append(row)

            added = len(seen_refs) - before
            print("New rows:", added)
            print("Total now:", len(seen_refs))

            # save after every page
            temp_df = pd.DataFrame(all_rows)
            for col in COLUMN_ORDER:
                if col not in temp_df.columns:
                    temp_df[col] = ""

            temp_df["Ref #"] = temp_df["Ref #"].astype(str)
            temp_df = (
                temp_df[COLUMN_ORDER]
                .drop_duplicates(subset=["Ref #"])
                .reset_index(drop=True)
            )

            temp_df.to_csv(EXISTING_FILE, index=False)
            temp_df.to_excel(DESKTOP / f"{file_base}.xlsx", index=False)

            if total_n and len(seen_refs) >= total_n:
                print("Finished all jobs.")
                break

            if end_n and total_n and end_n >= total_n:
                print("Reached final page.")
                break

            moved, _, status = await click_next_safe(page, end_n)
            print("Next status:", status)

            if status == "rate_limited":
                print("Rate limited after next click. Saving progress.")
                break

            if not moved:
                stale_count += 1
                if stale_count >= 2:
                    print("Pagination stopped twice. Ending run.")
                    break
            else:
                stale_count = 0
                page_num += 1

            await asyncio.sleep(6)

        final_df = pd.DataFrame(all_rows)
        for col in COLUMN_ORDER:
            if col not in final_df.columns:
                final_df[col] = ""

        final_df["Ref #"] = final_df["Ref #"].astype(str)
        final_df = (
            final_df[COLUMN_ORDER]
            .drop_duplicates(subset=["Ref #"])
            .reset_index(drop=True)
        )

        final_df.to_csv(EXISTING_FILE, index=False)
        final_df.to_excel(DESKTOP / f"{file_base}.xlsx", index=False)

        print("\nDONE")
        print("Final rows:", len(final_df))
        print("Saved CSV:", EXISTING_FILE)
        print("Saved XLSX:", DESKTOP / f"{file_base}.xlsx")

        return final_df

    finally:
        try:
            await context.close()
        except Exception:
            pass
        try:
            await browser.close()
        except Exception:
            pass
        try:
            await p.stop()
        except Exception:
            pass

# RUN RESUME VERSION
df = await scrape_gaswork_resume()
display(df.head())
print(df.shape)

Loading existing file...
Already have: 4155
Status: 200
URL: https://www.gaswork.com/search/Anesthesiologist/Job/All
Title: GasWork.com - Search - Anesthesiologist Jobs
Counter: (1, 25, 4283)
Trying to increase results per page...
Could not click 'Show 100 results per page': Locator.scroll_into_view_if_needed: Timeout 29984ms exceeded.
Call log:
  - attempting scroll into view action
    2 × waiting for element to be stable
      - element is not visible
    - retrying scroll into view action
    - waiting 20ms
    2 × waiting for element to be stable
      - element is not visible
    - retrying scroll into view action
      - waiting 100ms
    56 × waiting for element to be stable
       - element is not visible
     - retrying scroll into view action
       - waiting 500ms

Could not click '100 results per page': Locator.scroll_into_view_if_needed: Timeout 29985ms exceeded.
Call log:
  - attempting scroll into view action
    2 × waiting for element to be stable
      - element is n

,Compare,Ref #,City,State,Brief Description,Duration,Min $,Max $,W-2,User Type,Company,Updated,Date Posted,Job Link
0,NaN,317785,317785,Dallas,Texas,Pediatric Cardiac Anesthesiologist needed for ...,Full Time,"$700,000","$750,000",W-2,Group: National Private Practice,US Anesthesia Partners (USAP)- Medical City Da...,04/16/2026,https://www.gaswork.com/post/317785
1,NaN,334312,334312,"Frisco, Plano, and Dallas!",Texas,Seeking Cardiac or General full-time positions...,Full Time,"$500,000","$600,000",W-2,Group: National Private Practice,U.S. Anesthesia Partners- Excel Division,04/16/2026,https://www.gaswork.com/post/334312
2,NaN,553005,553005,Dallas,Texas,USAP is now looking for a Physician to serve a...,Full Time,NaN,NaN,W-2,Group: National Private Practice,U.S. Anesthesia Partners- Medical City Dallas-...,04/16/2026,https://www.gaswork.com/post/553005
3,NaN,372324,372324,Dallas,Texas,Customizable practice opportunity in Dallas! H...,Full Time,"$525,000",NaN,W-2,Group: National Private Practice,US Anesthesia Partners (USAP)- PAP Division,04/16/2026,https://www.gaswork.com/post/372324
4,NaN,431247,431247,San Antonio,Texas,100% Specialty or choose your own hybrid Peds/...,Full Time,NaN,NaN,W-2,Group: National Private Practice,US Anesthesia Partners (USAP),04/16/2026,https://www.gaswork.com/post/431247


(4269, 14)
